# Sanity Check: candidates_deduped

Basic validation and inspection of the deduplicated candidates dataset.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_parquet('../data/processed/candidates_deduped.parquet')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 7,032 rows × 11 columns


## 1. Schema & Data Types

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   problem_id   7032 non-null   object
 1   problem      7032 non-null   object
 2   answer       7032 non-null   object
 3   answer_type  7032 non-null   object
 4   source       7032 non-null   object
 5   difficulty   7032 non-null   object
 6   split        7032 non-null   object
 7   unit         7032 non-null   object
 8   tolerance    7032 non-null   object
 9   domain       7032 non-null   object
 10  language     7032 non-null   object
dtypes: object(11)
memory usage: 604.4+ KB


In [3]:
# Display dtypes more clearly
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

problem_id     object
problem        object
answer         object
answer_type    object
source         object
difficulty     object
split          object
unit           object
tolerance      object
domain         object
language       object
dtype: object

Memory usage: 7.65 MB


## 2. Nulls & Missing Values

In [4]:
null_counts = df.isnull().sum()
null_pct = 100 * df.isnull().sum() / len(df)

null_summary = pd.DataFrame({
    'column': null_counts.index,
    'null_count': null_counts.values,
    'null_pct': null_pct.values
}).sort_values('null_count', ascending=False)

print(null_summary[null_summary['null_count'] > 0])
if (null_summary['null_count'] == 0).all():
    print("✓ No nulls found!")

Empty DataFrame
Columns: [column, null_count, null_pct]
Index: []
✓ No nulls found!


## 3. Duplicates

In [5]:
dup_count = df.duplicated().sum()
dup_pct = 100 * dup_count / len(df)

print(f"Exact duplicates (all columns): {dup_count} ({dup_pct:.2f}%)")

if dup_count > 0:
    print("\nSample duplicate rows:")
    dup_rows = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)
    print(dup_rows)

Exact duplicates (all columns): 0 (0.00%)


## 4. Basic Statistics

In [6]:
df.describe(include='all').T

,count,unique,top,freq
problem_id,7032,7032,SciBench_RL_00426,1
problem,7032,7032,Suppose that a fair $n$-sided die is rolled $n...,1
answer,7032,6124,\boxed{0},98
answer_type,7032,151,NV,2010
source,7032,5,UGPhysics,5452
difficulty,7032,5,,6067
split,7032,1,train_candidate,7032
unit,7032,829,,4926
tolerance,7032,43,,6916
domain,7032,29,QuantumMechanics,999


## 5. Column-Specific Checks

In [7]:
# String columns: check for obvious issues
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique():,}")
    print(f"  Sample values: {df[col].unique()[:5]}")
    # Check for empty strings
    empty = (df[col] == '').sum()
    if empty > 0:
        print(f"  ⚠ Empty strings: {empty}")


problem_id:
  Unique values: 7,032
  Sample values: ['PHYSICS_00001' 'PHYSICS_00003' 'PHYSICS_00005' 'PHYSICS_00007'
 'PHYSICS_00009']

problem:
  Unique values: 7,032
  Sample values: ['The process of vaporization. At a pressure of $1.013 \\times 10^{5} \\mathrm{~Pa}$, how much does the internal energy of 1 mol of water increase when it turns into steam at $100^{\\circ} \\mathrm{C}$? It is known that at this pressure and temperature, the molar volumes of water and steam are $V_{1, \\mathrm{~m}}=18.8 \\mathrm{~cm}^{3} / \\mathrm{mol}$ and $V_{\\phi, \\mathrm{m}}=3.01 \\times 10^{4} \\mathrm{~cm}^{3} / \\mathrm{mol}$, respectively, and the latent heat of vaporization of water is $L=4.06 \\times 10^{4} \\mathrm{~J} / \\mathrm{mol}$.'
 'Entropy change of hot water. Place 1 kg of water at 20°C on a stove at 100°C to heat it, finally reaching 100°C. The specific heat of water is 4.18 × 10³ J/(kg · K). Calculate the entropy change of the water and the stove, ΔS_w and ΔS_l, respectively.'
 '

## 6. Sample Records (First 10)

In [8]:
df.head(10)

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,"[""\\boxed{1.01 \\times 10^{3}}"", ""\\boxed{-9.0...","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00005,Use white light as the light source to observe...,\boxed{1},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
3,PHYSICS_00007,"Under normal brightness, the diameter of the h...","[""\\boxed{2.24 \\times 10^{-4}}"", ""\\boxed{8.9}""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
4,PHYSICS_00009,Example 25.7: If an object is placed within th...,"[""\\boxed{-60}"", ""\\boxed{4}""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
5,PHYSICS_00011,Calculate the de Broglie wavelength of a bulle...,\boxed{2.21 \times 10^{-34}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Modern Physics,en
6,PHYSICS_00013,Consider a small bead with a mass of $m=1 \mat...,"[""\\boxed{1.05 \\times 10^{-33}}"", ""\\boxed{4....","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Modern Physics,en
7,PHYSICS_00015,To make a flat stone skip across the water sur...,\boxed{\frac{\sqrt{2 g h}}{\tan \theta}},Numerical,PHYSICS,High School and Below,train_candidate,,,Mechanics,en
8,PHYSICS_00017,Fishing boats commonly use echo sounders to em...,\boxed{B},MCQ,PHYSICS,High School and Below,train_candidate,,,Optics,en
9,PHYSICS_00019,"In a cylinder, a certain amount of ideal gas i...",\boxed{ABD},MCQ,PHYSICS,High School and Below,train_candidate,,,Thermodynamics,en


## 7. Sample Records (Random 10)

In [9]:
df.sample(min(10, len(df)))

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
2229,UGPhysics_ClassicalElectromagnetism_00358,For the electric field \( E_{l} \) of a family...,\boxed{\frac{\lambda}{2 \pi \varepsilon_{0}} \...,EX,UGPhysics,,train_candidate,,,ClassicalElectromagnetism,en
214,PHYSICS_00447,"In a cubic box with side length $L$, the parti...",1,"[""Numerical"", ""Numerical"", ""Numerical"", ""Numer...",PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Thermodynamics,en
3031,UGPhysics_ClassicalMechanics_00773,"A \(100 \, \mathrm{m}^2\) solar panel is conne...",\boxed{1.7 \times 10^{4}},NV,UGPhysics,,train_candidate,,,ClassicalMechanics,en
4491,UGPhysics_Relativity_00163,A rocket moves in a straight line with constan...,\boxed{\frac{4}{3a_{0}}},NV,UGPhysics,,train_candidate,,,Relativity,en
431,PHYSICS_00921,"In the double-slit interference experiment, mo...","[""0.11 \\, \\text{m}"", ""7""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
6566,OlympiadBench_OE_TO_physics_en_COMP_00149,1.1. Assume that the temperature of the atmosp...,$p(z)=p(0) e^{-\frac{\mu g}{R T_{0}} z}$,Expression,OlympiadBench,,train_candidate,,,OE_TO_physics_en_COMP,en
5852,UGPhysics_Thermodynamics_00098,There are large thermal reservoirs at $900 \ \...,\boxed{0},NV,UGPhysics,,train_candidate,,,Thermodynamics,en
6104,UGPhysics_Thermodynamics_00354,"In a physics experiment, a large solenoid is m...",\boxed{2.49 \times 10^{4}},NV,UGPhysics,,train_candidate,,,Thermodynamics,en
4107,UGPhysics_QuantumMechanics_00794,Prove that the full width at half maximum (FWH...,\boxed{\gamma},NV,UGPhysics,,train_candidate,,,QuantumMechanics,en
4824,UGPhysics_Solid-StatePhysics_00109,Given the previously derived results \(I_{1} =...,\boxed{\frac{16}{\left(4+a_{0}^{2} G^{2}\right...,EX,UGPhysics,,train_candidate,,,Solid-StatePhysics,en


In [10]:
df

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,"[""\\boxed{1.01 \\times 10^{3}}"", ""\\boxed{-9.0...","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00005,Use white light as the light source to observe...,\boxed{1},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
3,PHYSICS_00007,"Under normal brightness, the diameter of the h...","[""\\boxed{2.24 \\times 10^{-4}}"", ""\\boxed{8.9}""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
4,PHYSICS_00009,Example 25.7: If an object is placed within th...,"[""\\boxed{-60}"", ""\\boxed{4}""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
...,...,...,...,...,...,...,...,...,...,...,...
7027,SciBench_RL_00422,An urn contains four colored balls: two orange...,0.2,numerical,SciBench_RL,,train_candidate,,,stat,en
7028,SciBench_RL_00423,"Bowl $B_1$ contains two white chips, bowl $B_2...",0.65625,numerical,SciBench_RL,,train_candidate,,,stat,en
7029,SciBench_RL_00424,Divide a line segment into two parts by select...,0.66666666666,numerical,SciBench_RL,,train_candidate,,,stat,en
7030,SciBench_RL_00425,"In a state lottery, four digits are drawn at r...",0.0024,numerical,SciBench_RL,,train_candidate,,,stat,en


In [11]:
df.iloc[7031]['problem']

'Suppose that a fair $n$-sided die is rolled $n$ independent times. A match occurs if side $i$ is observed on the $i$ th trial, $i=1,2, \\ldots, n$. Find the limit of this probability as $n$ increases without bound.'

In [12]:
df[df['problem_id'] == 'PHYSICS_00081']['problem'].values[0]

'A homogeneous rod $AB$, with mass $m$ and length $2a$, has its end $A$ moving along a smooth horizontal groove. The rod itself can swing in a vertical plane around end $A$. In addition to the force of gravity, end $B$ is subject to a horizontal force $F$. Use the Lagrange equation to derive its equation of motion. What if the angle of swing is very small?'

In [13]:
df[df['problem_id'] == 'PHYSICS_00081']['answer'].values[0]

'["m(\\\\ddot{x}+a \\\\ddot{\\\\theta} \\\\cos \\\\theta-a \\\\dot{\\\\theta}^{2} \\\\sin \\\\theta)=F", "\\\\ddot{x}+a \\\\ddot{\\\\theta}=\\\\frac{F}{m}"]'

In [14]:
df[df['problem_id'] == 'PHYSICS_00081']['answer_type'].values

array(['["Equation", "Equation", "Equation", "Equation"]'], dtype=object)

In [15]:
df[df['problem_id'] == 'PHYSICS_00087']['problem'].values[0]

"Observers O and $\\mathrm{O}^{\\prime}$ are approaching each other at a relative velocity of $0.6 c$. If O measures the initial distance of $\\mathrm{O}^{\\prime}$ from him to be 20 m, according to O's measurement, how long will it take to meet $\\mathrm{O}^{\\prime}$? According to $\\mathrm{O}^{\\prime}$'s measurement, how long will it take to meet O?"

In [22]:
df[df['problem_id'] == 'PHYSICS_00087']['answer'].values[0]

'1.11 \\times 10^{-7} \\, \\text{s}'

In [16]:
df[df['source'] == 'PHYSICS']

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,"[""\\boxed{1.01 \\times 10^{3}}"", ""\\boxed{-9.0...","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00005,Use white light as the light source to observe...,\boxed{1},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
3,PHYSICS_00007,"Under normal brightness, the diameter of the h...","[""\\boxed{2.24 \\times 10^{-4}}"", ""\\boxed{8.9}""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
4,PHYSICS_00009,Example 25.7: If an object is placed within th...,"[""\\boxed{-60}"", ""\\boxed{4}""]","[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
...,...,...,...,...,...,...,...,...,...,...,...
960,PHYSICS_01990,Find the velocity $v$ of a particle with mass ...,\sqrt{\frac{2 e V}{m}} \left(1-\frac{3}{4} \fr...,"[""Expression"", ""Expression""]",PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Modern Physics,en
961,PHYSICS_01992,A layer of a dielectric with a volume charge d...,2 \pi \rho d \frac{\vec{z}}{|z|},"[""Expression"", ""Expression""]",PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Electromagnetism,en
962,PHYSICS_01994,Let a circuit be composed of two coils joined ...,L_1 + L_2 - 2\mathfrak{M},"[""Expression"", ""Expression""]",PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Electromagnetism,en
963,PHYSICS_01996,A plane-polarized wave falls normally on the s...,\frac{(1-n)^{2}+\kappa^{2}}{(1+n)^{2}+\kappa^{2}},Expression,PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Electromagnetism,en


In [21]:
df[df['problem_id'] == 'PHYSICS_00115']['problem'].values[0]

'A Fabry-Pérot resonator (abbreviated as Fabry-Pérot cavity) is 5 cm long, and an extended light source is used in the experiment with a light wavelength of $0.6 \\mu \\mathrm{~m}$. Questions:\n\n(1) What is the central interference order?\n\n(2) What is the half-angular width of the interference ring near an inclination angle of $1^{\\circ}$? (Assume the reflectance $R=0.98$)\n\n(3) If this Fabry-Pérot cavity is used to resolve spectral lines, what is its spectral resolving power? What is the smallest resolvable wavelength interval?\n\n(4) If this Fabry-Pérot cavity is used for frequency selection of white light, how many spectral lines are transmitted most strongly? What is the width of each spectral line?\n\n(5) Due to thermal expansion and contraction, the change in cavity length is $10^{-5}$ (relative value). What is the amount of spectral line shift?'

In [24]:
df[df['problem_id'] == 'PHYSICS_00115']['answer'].values[0]

'["1.7 \\\\times 10^{5}", "0.45^{\\\\prime \\\\prime}", "2.6 \\\\times 10^{7}", "1.2 \\\\times 10^{5}", "3 \\\\times 10^{4} \\\\mathrm{~Hz}"]'

In [25]:
df[df['problem_id'] == 'PHYSICS_00115']['answer_type'].values[0]

'["Numerical", "Numerical", "Numerical", "Numerical", "Numerical", "Numerical", "Numerical", "Numerical"]'

## 8. Inspect Individual Records (Interactive)

In [17]:
# View a specific record with nice formatting
idx = 0  # Change this to inspect different rows
print(f"\nRecord #{idx}:")
print("=" * 80)
for col, val in df.iloc[idx].items():
    print(f"{col:30s} : {val}")


Record #0:
problem_id                     : PHYSICS_00001
problem                        : The process of vaporization. At a pressure of $1.013 \times 10^{5} \mathrm{~Pa}$, how much does the internal energy of 1 mol of water increase when it turns into steam at $100^{\circ} \mathrm{C}$? It is known that at this pressure and temperature, the molar volumes of water and steam are $V_{1, \mathrm{~m}}=18.8 \mathrm{~cm}^{3} / \mathrm{mol}$ and $V_{\phi, \mathrm{m}}=3.01 \times 10^{4} \mathrm{~cm}^{3} / \mathrm{mol}$, respectively, and the latent heat of vaporization of water is $L=4.06 \times 10^{4} \mathrm{~J} / \mathrm{mol}$.
answer                         : \boxed{3.75 \times 10^{4}}
answer_type                    : Numerical
source                         : PHYSICS
difficulty                     : Undergraduate (Non-Physics Major),
split                          : train_candidate
unit                           : 
tolerance                      : 
domain                         : Thermod

## 9. Summary

In [18]:
print(f"✓ Total records: {len(df):,}")
print(f"✓ Total columns: {len(df.columns)}")
print(f"✓ Columns: {', '.join(df.columns.tolist())}")
print(f"✓ Null values: {df.isnull().sum().sum()}")
print(f"✓ Duplicates: {df.duplicated().sum()}")

✓ Total records: 7,032
✓ Total columns: 11
✓ Columns: problem_id, problem, answer, answer_type, source, difficulty, split, unit, tolerance, domain, language
✓ Null values: 0
✓ Duplicates: 0
